# Step 5: Fine-Tuning EoMT with LoRA on Cityscapes - Three Experiments

In this step we fine-tune an EoMT model pre-trained on COCO to the Cityscapes semantic segmentation task. Rather than updating all 86 M backbone parameters, we inject **LoRA adapters** (Hu et al., 2022) into the ViT attention projections and then choose, per experiment, which subset of those adapters is actually allowed to train.

All three experiments use `MaskClassificationLoRA` and the same LoRA config.
The only difference between them is which parameters have `requires_grad=True`.
`configure_optimizers` adapts automatically: it filters by `requires_grad` and
routes whatever is trainable into the right param groups for
`TwoStageWarmupPolySchedule` + LLRD.

| Experiment | Trainable |
|---|---|
| 1 - head only | prediction head; all LoRA adapters frozen |
| 2 - decoder LoRA | prediction head + LoRA adapters in blocks 9–11 |
| 3 - full LoRA | prediction head + LoRA adapters in all 12 blocks (LLRD) |

## 1. Environment Setup

In [5]:
!pip install lightning gitignore_parser peft > /dev/null
!pip install -U "torchao>=0.16.0" > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' > /dev/null
!pip install wandb > /dev/null

## 2. Imports

In [ ]:
from google.colab import drive, userdata
import os, sys, json, yaml, glob, shutil, time, random, re
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm
from lightning import seed_everything
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
from collections import defaultdict, Counter
from torch.utils.data import DataLoader, Subset
import pandas as pd

## 3. Mount Drive and Configure Paths

In [6]:
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
eomt_folder  = project_root + '/eomt'

if not os.path.exists('/content/ProjectFolder'):
    os.symlink(project_root, '/content/ProjectFolder')

os.chdir(project_root)
for p in [project_root, eomt_folder]:
    if p not in sys.path:
        sys.path.insert(0, p)

### 3a. Project-Specific Imports and Safe Globals

In [ ]:
from eval.iouEval import iouEval
from training.mask_classification_panoptic import MaskClassificationPanoptic
from training.mask_classification_lora import MaskClassificationLoRA
from models.eomt import EoMT
from models.vit import ViT
from datasets.cityscapes_semantic import CityscapesSemantic
import torch.serialization
from peft import PeftModel
from peft.tuners.lora import LoraModel

# Whitelist the classes that get pickled into our .ckpt files.
# Without this, torch.load with weights_only=True raises an UnpicklingError.
torch.serialization.add_safe_globals([EoMT, ViT, MaskClassificationLoRA, PeftModel, LoraModel, range])

seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 4. Copy Dataset to Local SSD

In [ ]:
drive_data = os.path.join(eomt_folder, 'data')
local_data = '/content/cityscapes_data'
os.makedirs(local_data, exist_ok=True)

for fname in ['leftImg8bit_trainvaltest.zip', 'gtFine_trainvaltest.zip']:
    src = os.path.join(drive_data, fname)
    dst = os.path.join(local_data, fname)
    if not os.path.exists(dst):
        print(f"Copying {fname} to local SSD...")
        t = time.time()
        shutil.copy(src, dst)
        print(f"  Done in {time.time()-t:.0f}s ({os.path.getsize(dst)/1e9:.1f} GB)")
    else:
        print(f"  {fname} already on local SSD")

data_path = local_data
print(f"data_path → {data_path}")

Copying leftImg8bit_trainvaltest.zip to local SSD...
  Done in 285s (11.6 GB)
Copying gtFine_trainvaltest.zip to local SSD...
  Done in 9s (0.3 GB)
data_path → /content/cityscapes_data


## 5. Helper Functions

In [ ]:
def load_cfg(filename):
    """Load a YAML experiment config from configs/experiments/."""
    path = os.path.join(eomt_folder, "configs", "experiments", filename)
    with open(path) as f:
        cfg = yaml.safe_load(f)
    return cfg


def build_lora_model(cfg, trainable_blocks=None):
    """
    Instantiate a MaskClassificationLoRA model.
 
    `trainable_blocks` controls which ViT blocks have their LoRA adapters
    unfrozen. Passing an empty list (or None) keeps all adapters frozen:
    only the prediction head trains. Passing range(12) unfreezes all of them.
    """
    encoder_ft = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
    network_ft = EoMT(num_classes=19, encoder=encoder_ft,
                       num_q=cfg.get("num_q", 200), num_blocks=3, masked_attn_enabled=False)
    return MaskClassificationLoRA(
        network=network_ft,
        img_size=(640, 640),
        num_classes=19,
        attn_mask_annealing_enabled=cfg.get("attn_mask_annealing_enabled", False),
        attn_mask_annealing_start_steps=cfg.get("attn_mask_annealing_start_steps", None),
        attn_mask_annealing_end_steps=cfg.get("attn_mask_annealing_end_steps", None),
        lr=cfg["lr"],
        llrd=cfg["llrd"],
        weight_decay=cfg["weight_decay"],
        poly_power=cfg["poly_power"],
        warmup_steps=cfg["warmup_steps"],
        ckpt_path=bin_path,
        load_ckpt_class_head=False, # head weights don't transfer from COCO --> Cityscapes
        lora_r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_target_modules=cfg["lora_target_modules"],
        lora_dropout=cfg["lora_dropout"],
        trainable_blocks=trainable_blocks,
    )


def build_trainer(cfg):
    """
    Build a Lightning Trainer with W&B logging and checkpoint saving.
 
    We call wandb.finish() first to cleanly close any run that might still
    be open from a previous cell execution (common when iterating in Colab).
    The ModelCheckpoint monitors val_loss_total in 'min' mode rather than
    mIoU because mIoU requires the full evaluation loop and would be too
    slow to compute at every validation step during training.
    """
    wandb.finish()
    if wandb.run is None:
        try:
            from google.colab import userdata
            wandb.login(key=userdata.get("WANDDB-API-KEY"))
        except (ImportError, Exception):
            wandb.login()

    run_name = cfg["experiment_name"]
    return Trainer(
        max_epochs=cfg["max_epochs"],
        accelerator="auto",
        devices=1,
        precision="16-mixed",
        accumulate_grad_batches=cfg.get("accumulate_grad_batches", 1),
        gradient_clip_val=cfg.get("gradient_clip_val", 0.01),
        gradient_clip_algorithm="norm",
        log_every_n_steps=10,
        num_sanity_val_steps=0,
        logger=WandbLogger(project="eomt-cityscapes-finetuning",
                           name=run_name,
                           resume = "allow"), # resume the W&B run if the experiment name matches
        callbacks=[
            LearningRateMonitor(logging_interval="step"),
            ModelCheckpoint(
                dirpath=os.path.join(project_root, "checkpoints", run_name),
                filename="eomt-{epoch:02d}-{losses/val_loss_total:.3f}",
                save_top_k=2,
                monitor="losses/val_loss_total", 
                mode="min",                      
                save_last=True,
            ),
        ],
    )


def find_latest_ckpt(run_name):
    """
    Return the path of the most recent checkpoint for this run, or None
    if no checkpoint exists yet (i.e. training is starting from scratch).
    Prefers last.ckpt over epoch checkpoints so that the optimizer state
    and scheduler steps are also restored correctly.
    """
    ckpt_dir = os.path.join(project_root, "checkpoints", run_name)
    last = os.path.join(ckpt_dir, "last.ckpt")
    if os.path.exists(last):
        print(f"Resuming from: {last}")
        return last
    ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.ckpt")))
    if ckpts:
        print(f"Resuming from: {ckpts[-1]}")
        return ckpts[-1]
    print(f"No checkpoint for '{run_name}', starting from scratch.")
    return None


def print_trainable(model):
    """Print trainable vs total parameter counts as a quick sanity check."""

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

 ## 6. Dataset Preparation
 Cityscapes training images come from 18 different cities. A naive random split risks over-representing some cities in the validation fold, which would make val-loss curves noisy and less representative of generalisation performance
 Instead we do a **city-stratified split**: for each city independently we
 randomly assign 80% of its images to `train_idx` and the remaining 20% to `val_idx`. This ensures that every city appears in both folds with the correct proportion and that the val-loss curve tracks generalisation rather than city-specific variance.

The split is performed over the 2975 official training images only. The 500 official validation images are left completely untouched until the final hold-out evaluation in Section 7.

In [ ]:
def city_from_path(path):
    """Extract the city name from a Cityscapes filename (first underscore-delimited token)."""
    return os.path.basename(str(path)).split("_")[0]


def get_city_labels(dataset, data_path):
    """
    Return a list of city strings aligned with dataset indices.
 
    We try several common attribute names used by different torchvision /
    custom dataset implementations. If none of them exist we fall back to
    scanning the leftImg8bit/train directory on disk — slower but always
    correct.
    """
    for attr in ("imgs", "files", "images", "samples", "_images", "img_paths"):
        raw = getattr(dataset, attr, None)
        if raw is not None:
            return [city_from_path(e[0] if isinstance(e, (list, tuple)) else e)
                    for e in raw]
    
    # Fallback: walk the train directory and count PNGs per city folder    train_img_root = os.path.join(data_path, "leftImg8bit", "train")
    labels = []
    for city in sorted(os.listdir(train_img_root)):
        city_dir = os.path.join(train_img_root, city)
        if not os.path.isdir(city_dir):
            continue
        n = len([f for f in os.listdir(city_dir) if f.endswith(".png")])
        labels.extend([city] * n)
    assert len(labels) == len(dataset), (
        f"City label count ({len(labels)}) != dataset size ({len(dataset)}). "
        "Check that data_path points to the correct Cityscapes root."
    )
    return labels


# Load the shared config (batch size, augmentation flags, etc.) once.
cfg_shared = load_cfg('optimal_lora_config.yaml')
dm_train = CityscapesSemantic(
    path=data_path,
    batch_size=cfg_shared['batch_size'],
    num_workers=4,
    img_size=(640, 640),
    gaussian_blur_enabled=cfg_shared.get('gaussian_blur_enabled', False),
)
dm_train.setup('fit')

train_full  = dm_train.train_dataloader().dataset
city_labels = get_city_labels(train_full, data_path)

# City-stratified 80/20 split 
# Split within each city so every city is proportionally represented in both sets.
rng = random.Random(0) # Seed the RNG with a fixed value so the split is reproducible across runs.
train_idx, val_idx = [], []
for city, idxs in sorted(defaultdict(list, {
        c: [i for i, l in enumerate(city_labels) if l == c]
        for c in set(city_labels)}).items()):
    shuffled = list(idxs); rng.shuffle(shuffled)
    n_val = max(1, round(len(shuffled) * 0.2)) # at least one val image per city
    val_idx.extend(shuffled[:n_val])
    train_idx.extend(shuffled[n_val:])

print(f"Train: {len(train_idx)} images | Val-loss: {len(val_idx)} images")

# Print the per-city breakdown so we can verify that every city has val images.
train_counts = Counter(city_labels[i] for i in train_idx)
val_counts   = Counter(city_labels[i] for i in val_idx)
print("\nPer-city split (train / val):")
for city in sorted(train_counts):
    print(f"  {city:<20} {train_counts[city]:>4} / {val_counts.get(city, 0):>3}")

# Dataloaders 
# Inherit the collate function from the original datamodule so that the
# variable-resolution images are handled correctly during evaluation.
_collate = getattr(dm_train.train_dataloader(), 'collate_fn', None)
bs = cfg_shared['batch_size']

train_loader = DataLoader(
    Subset(train_full, train_idx),
    batch_size=bs,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True, # drop the last incomplete batch to keep batch stats stable
    collate_fn=_collate,
)
val_loss_loader = DataLoader(
    Subset(train_full, val_idx),
    batch_size=bs,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=_collate,
)

## 7. Training Experiments

Three experiments with increasing numbers of trainable LoRA blocks.
All share the same `CityscapesSemantic` train/val loaders and `TwoStageWarmupPolySchedule`.

**No data leakage**: training and val-loss monitoring use **only the 80/20 split of
the 2975 Cityscapes training images**. The official 500-image val set is never touched
here — it is the held-out test set used exclusively in Section 5.

### Experiment 1 - Head Only
Prediction head and query embeddings train; all LoRA adapters are frozen.

In [ ]:
bin_path = os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin')
cfg1    = load_cfg("exp1_lora_head_only.yaml")
model1  = build_lora_model(cfg1, trainable_blocks=[]) # empty list --> all adapters frozen
print_trainable(model1)
trainer1 = build_trainer(cfg1)
trainer1.fit(
    model=model1,
    train_dataloaders=train_loader,
    val_dataloaders=val_loss_loader,
    ckpt_path=find_latest_ckpt(cfg1["experiment_name"]),
)

### Experiment 2 - Decoder LoRA (blocks 9–11)
Prediction head + LoRA adapters in the last 3 decoder blocks (9, 10, 11) train.

In [ ]:
cfg2    = load_cfg("exp2_lora_decoder_blocks.yaml")
model2  = build_lora_model(cfg2, trainable_blocks=range(9, 12))
print_trainable(model2)
trainer2 = build_trainer(cfg2)
trainer2.fit(
    model=model2,
    train_dataloaders=train_loader,
    val_dataloaders=val_loss_loader,
    ckpt_path=find_latest_ckpt(cfg2["experiment_name"]),
)

### Experiment 3 - Full LoRA (all 12 blocks)
Prediction head + LoRA adapters in all 12 blocks train with LLRD (layer-wise LR decay).

In [ ]:
bin_path = os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin')
cfg3    = load_cfg("exp3_lora_all_blocks.yaml")
model3  = build_lora_model(cfg3, trainable_blocks=range(12))
print_trainable(model3)
trainer3 = build_trainer(cfg3)
trainer3.fit(
    model=model3,
    train_dataloaders=train_loader,
    val_dataloaders=val_loss_loader,
    ckpt_path=find_latest_ckpt(cfg3["experiment_name"]),
)

## 8. Final Hold-Out Evaluation

> **⚠️ Run this section only once, after hyperparameter search is complete.**
>
> The 500-image official Cityscapes val set is used **exclusively here** as a
> held-out test set. It has never been seen during training or val-loss monitoring
> (Section 4 uses only the 80/20 split of the 2975 training images).
> Running it mid-search would leak test information into your config decisions.

mIoU is computed with windowed semantic inference, mirroring `eval_step`.

In [ ]:
CS_CLASSES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain",
    "sky", "person", "rider", "car", "truck", "bus",
    "train", "motorcycle", "bicycle",
]


def evaluate_miou(model, data_path, device, num_classes=19, desc="eval"):
    """mIoU on the official Cityscapes val set (500 images), mirrors eval_step."""
    dm_eval = CityscapesSemantic(path=data_path, batch_size=1, num_workers=4, img_size=(640, 640))
    dm_eval.setup()
    loader = dm_eval.val_dataloader()
    model.eval().to(device)
    ev = iouEval(num_classes + 1)
    for batch in tqdm(loader, desc=desc):
        imgs, targets = batch
        imgs = [img.to(device) for img in imgs]   # eval_collate returns a tuple of tensors, not a stacked tensor
        gt   = model.to_per_pixel_targets_semantic(targets, model.ignore_idx)
        img_sizes = [img.shape[-2:] for img in imgs]
        with torch.no_grad():
            crops, origins = model.window_imgs_semantic(imgs)
            mlp, clp = model(crops)
            ml     = F.interpolate(mlp[-1], model.img_size, mode="bilinear")
            logits = model.to_per_pixel_logits_semantic(ml, clp[-1])
            preds  = model.revert_window_logits_semantic(logits, origins, img_sizes)  # list of (C,H,W) tensors
        for pred, g in zip(preds, gt):
            # Remap the ignore_idx (usually 255) to num_classes (19) so it fits in the iouEval bins (0 to 19)
            g_mapped = g.clone()
            g_mapped[g_mapped == model.ignore_idx] = num_classes

            ev.addBatch(pred.argmax(dim=0).unsqueeze(0).unsqueeze(0),   # dim=0: C is axis 0 in (C,H,W)
                        g_mapped.unsqueeze(0).unsqueeze(0))
    _, ious = ev.getIoU()
    miou = ious[:num_classes].mean().item() * 100
    return miou, (ious[:num_classes] * 100).tolist()

def get_top_checkpoints_from_disk(run_name):
    """
    Find the top-2 checkpoints for a given run, ranked by val_loss_total
    extracted from the filename. last.ckpt files are excluded because they
    are not guaranteed to correspond to the best validation performance.
    """
    ckpt_dir = os.path.join(project_root, "checkpoints", run_name)
    if not os.path.exists(ckpt_dir):
        return []
    # Find all checkpoints recursively
    ckpts = glob.glob(os.path.join(ckpt_dir, "**", "*.ckpt"), recursive=True)

    ckpt_scores = []
    for ckpt in ckpts:
        # Skip the generic last checkpoints
        if ckpt.endswith("last.ckpt") or ckpt.endswith("last-v1.ckpt"):
            continue

        match = re.search(r'val_loss_total=([0-9]+\.[0-9]+)', ckpt)
        if match:
            score = float(match.group(1))
            ckpt_scores.append((score, ckpt))

    # Sort by validation loss (ascending) and return top 2
    ckpt_scores.sort(key=lambda x: x[0])
    return [c[1] for c in ckpt_scores[:2]]

eval_results = {}

def eval_top_k_for_exp(model, run_name, exp_name):
    """
    Load the top-K checkpoints for an experiment and evaluate each against the
    held-out val set, storing results in the global `eval_results` dict with
    keys like 'Exp 1_top1', 'Exp 1_top2'.
 
    If no checkpoints are found on disk (e.g. the experiment is still running)
    the current model weights are evaluated instead and stored under '_current'.
    """
    ckpts = get_top_checkpoints_from_disk(run_name)
    if not ckpts:
        print(f"[{exp_name}] No checkpoints found for {run_name} on disk, using current weights.")
        miou, ious = evaluate_miou(model, data_path, device, desc=f"{exp_name} - Current")
        eval_results[f"{exp_name}_current"] = {"miou": miou, "ious": ious}
        return

    for i, ckpt_path in enumerate(ckpts):
        print(f"[{exp_name}] Loading Top {i+1} checkpoint from: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=model.device)
        model.load_state_dict(ckpt['state_dict'])
        desc = f"{exp_name} - Top {i+1}"
        miou, ious = evaluate_miou(model, data_path, device, desc=desc)
        eval_results[f"{exp_name}_top{i+1}"] = {"miou": miou, "ious": ious}


In [ ]:
# --- Re-initialize models and trainers ---
local_data = '/content/cityscapes_data'
data_path = local_data

if 'eomt_folder' not in globals():
    eomt_folder = '/content/drive/MyDrive/FundGitHubProject/eomt'


# Left because is needed in the function build_lora_model
bin_path = os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin')

cfg1 = load_cfg("exp1_lora_head_only.yaml")
model1 = build_lora_model(cfg1, trainable_blocks=[])

cfg2 = load_cfg("exp2_lora_decoder_blocks.yaml")
model2 = build_lora_model(cfg2, trainable_blocks=range(9, 12))

cfg3 = load_cfg("exp3_lora_all_blocks.yaml")
model3 = build_lora_model(cfg3, trainable_blocks=range(12))
# -----------------------------------------

# Evaluate all models by fetching checkpoints from the disk
torch.cuda.empty_cache()
eval_top_k_for_exp(model1, cfg1["experiment_name"], "Exp 1")
eval_top_k_for_exp(model2, cfg2["experiment_name"], "Exp 2")
eval_top_k_for_exp(model3, cfg3["experiment_name"], "Exp 3")

 ## 9. Scoreboard and Checkpoint Export

In [ ]:
# Create a dedicated directory outside of 'checkpoints'
export_dir = os.path.join(project_root, "best_eval_checkpoints")
os.makedirs(export_dir, exist_ok=True)

# Create a DataFrame for the scoreboard
records = []
for exp_key, data in eval_results.items():
    records.append({
        "Experiment": exp_key.split('_')[0],
        "Variant": exp_key.split('_')[1],
        "mIoU": data["miou"]
    })

df_scores = pd.DataFrame(records).sort_values(by="mIoU", ascending=False).reset_index(drop=True)

print("=== FINAL SCOREBOARD ===")
display(df_scores)

# Save scoreboard to the dedicated folder
csv_path = os.path.join(export_dir, "scoreboard_miou.csv")
df_scores.to_csv(csv_path, index=False)
print(f"\nScoreboard saved to: {csv_path}\n")

# Map experiments to their original checkpoint paths on disk
ckpt_paths = {
    "Exp 1": get_top_checkpoints_from_disk(cfg1["experiment_name"]),
    "Exp 2": get_top_checkpoints_from_disk(cfg2["experiment_name"]),
    "Exp 3": get_top_checkpoints_from_disk(cfg3["experiment_name"]),
}

# Identify Winners
overall_winner_exp = df_scores.iloc[0]["Experiment"]
overall_winner_var = df_scores.iloc[0]["Variant"]

# Copy checkpoints with distinctive names
print("=== SAVING CHECKPOINTS ===")
for exp_name in ["Exp 1", "Exp 2", "Exp 3"]:
    # Find winner for this specific experiment
    exp_df = df_scores[df_scores["Experiment"] == exp_name]
    if exp_df.empty:
        continue
    exp_winner_var = exp_df.iloc[0]["Variant"]

    paths = ckpt_paths.get(exp_name, [])

    for i, ckpt_path in enumerate(paths):
        var_name = f"top{i+1}"
        key = f"{exp_name}_{var_name}"
        if key not in eval_results:
            continue

        miou_score = eval_results[key]["miou"]
        exp_str = exp_name.replace(" ", "")

        # Default distinctive naming
        new_name = f"{exp_str}_{var_name}_mIoU_{miou_score:.2f}.ckpt"

        # Check if it's the experiment winner
        if var_name == exp_winner_var:
            new_name = f"WINNER_{exp_str}_mIoU_{miou_score:.2f}.ckpt"

        # Check if it's the OVERALL winner (Overwrites experiment winner naming)
        if exp_name == overall_winner_exp and var_name == overall_winner_var:
            new_name = f"OVERALL_CHAMPION_{exp_str}_mIoU_{miou_score:.2f}.ckpt"

        dest_path = os.path.join(export_dir, new_name)
        shutil.copy2(ckpt_path, dest_path)
        print(f"Copied {exp_name} {var_name} -> {new_name}")

print(f"\nAll checkpoints have been securely backed up in: {export_dir}")


=== FINAL SCOREBOARD ===

Experiment Variant       mIoU

0      Exp 3    top1  75.387542

1      Exp 3    top2  74.741934

2      Exp 2    top2  73.311495

3      Exp 2    top1  73.245060

4      Exp 1    top2  72.014649

5      Exp 1    top1  71.877525

Scoreboard saved to: /content/drive/MyDrive/FundGitHubProject/best_eval_checkpoints/scoreboard_miou.csv

=== SAVING CHECKPOINTS ===

Copied Exp 1 top1 -> Exp1_top1_mIoU_71.88.ckpt

Copied Exp 1 top2 -> WINNER_Exp1_mIoU_72.01.ckpt

Copied Exp 2 top1 -> Exp2_top1_mIoU_73.25.ckpt

Copied Exp 2 top2 -> WINNER_Exp2_mIoU_73.31.ckpt

Copied Exp 3 top1 -> OVERALL_CHAMPION_Exp3_mIoU_75.39.ckpt

Copied Exp 3 top2 -> Exp3_top2_mIoU_74.74.ckpt

All checkpoints have been securely backed up in: /content/drive/MyDrive/FundGitHubProject/best_eval_checkpoints

 ## 10. Checkpoint Ranking by Validation Loss

In [ ]:
def find_best_validation_scores(project_root):
    """
    Scan all checkpoint directories recursively and print the top-10
    checkpoints ranked by val_loss_total extracted from the filename.
    """
    checkpoint_dir = os.path.join(project_root, "checkpoints")
    # Search recursively for all .ckpt files
    all_ckpts = glob.glob(os.path.join(checkpoint_dir, "**", "*.ckpt"), recursive=True)

    ckpt_scores = []
    for ckpt in all_ckpts:
        # Skip the generic last.ckpt files
        if ckpt.endswith("last.ckpt") or ckpt.endswith("last-v1.ckpt"):
            continue

        # Use regex to find the float value after 'val_loss_total='
        # The filename format is typically eomt-epoch=XX-losses/val_loss_total=YY.YYY.ckpt
        # We use ([0-9]+\.[0-9]+) to avoid matching the trailing dot from .ckpt
        match = re.search(r'val_loss_total=([0-9]+\.[0-9]+)', ckpt)
        if match:
            score = float(match.group(1))
            ckpt_scores.append((score, ckpt))

    if not ckpt_scores:
        print("No valid checkpoints with 'val_loss_total' found in the filenames.")
        return

    # Sort by the extracted validation loss (ascending)
    ckpt_scores.sort(key=lambda x: x[0])

    print("Top Checkpoints by Validation Loss:")
    print("=" * 80)
    for rank, (score, ckpt) in enumerate(ckpt_scores[:10], start=1):
        # Extract just the experiment name and filename for cleaner printing
        parts = ckpt.split('/')
        exp_name = parts[-3] if len(parts) >= 3 else "Unknown"
        filename = parts[-1]
        print(f"{rank}. Loss: {score:.4f} | Exp: {exp_name:<20} | File: {filename}")

find_best_validation_scores(project_root)


Top Checkpoints by Validation Loss:
================================================================================
1. Loss: 2.0090 | Exp: lora-all-blocks      | File: val_loss_total=2.009.ckpt
2. Loss: 2.0100 | Exp: lora-all-blocks      | File: val_loss_total=2.010.ckpt
3. Loss: 2.1380 | Exp: lora-decoder-blocks  | File: val_loss_total=2.138.ckpt
4. Loss: 2.1380 | Exp: lora-decoder-blocks  | File: val_loss_total=2.138.ckpt
5. Loss: 2.2390 | Exp: lora-head-only       | File: val_loss_total=2.239.ckpt
6. Loss: 2.2420 | Exp: lora-head-only       | File: val_loss_total=2.242.ckpt
7. Loss: 2.2450 | Exp: lora-head-only       | File: val_loss_total=2.245.ckpt
8. Loss: 2.2630 | Exp: lora-head-only       | File: val_loss_total=2.263.ckpt
9. Loss: 2.2810 | Exp: lora-head-only       | File: val_loss_total=2.281.ckpt
10. Loss: 2.2900 | Exp: lora-head-only       | File: val_loss_total=2.290.ckpt